# Train the English-to-French model

This notebook orchestrates the reusable package code. Install the project with `python -m pip install -e ".[dev]"` before running it.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import torch

from lstm_translator import (
    BPETokenizer,
    Seq2Seq,
    Seq2SeqConfig,
    TrainingConfig,
    Translator,
    Vocabulary,
    load_parallel_tsv,
    save_checkpoint,
    split_parallel,
    train_model,
    translation_scores,
)

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {DEVICE}")

## Load and tokenize the corpus

In [ ]:
source_tokenizer = BPETokenizer.load(ROOT / "artifacts/tokenizers/en.json")
target_tokenizer = BPETokenizer.load(ROOT / "artifacts/tokenizers/fr.json")
english, french = load_parallel_tsv(ROOT / "data/raw/fra.txt")
train_english, train_french, test_english, test_french = split_parallel(
    english, french, test_fraction=0.1, seed=42
)
source_sequences = source_tokenizer.encode_batch(train_english)
target_sequences = target_tokenizer.encode_batch(train_french)
print(f"Training examples: {len(train_english):,}; test examples: {len(test_english):,}")

## Configure the model and training loop

In [ ]:
source_vocabulary = Vocabulary(source_tokenizer.tokens)
target_vocabulary = Vocabulary(target_tokenizer.tokens)
model_config = Seq2SeqConfig(
    hidden_size=128,
    num_layers=4,
    embedding_dim=42,
    dropout=0.1,
    attention=True,
)
training_config = TrainingConfig(
    epochs=40,
    batch_size=32,
    learning_rate=1e-3,
    teacher_forcing_start=1.0,
    teacher_forcing_end=0.1,
    teacher_forcing_decay_epochs=25,
    patience=5,
    seed=42,
)
model = Seq2Seq(
    len(source_vocabulary),
    len(target_vocabulary),
    model_config,
).to(DEVICE)

In [ ]:
checkpoint_path = ROOT / "artifacts/checkpoints/model-v1.pth"

def save_best(current_model, metrics):
    save_checkpoint(
        checkpoint_path,
        current_model,
        source_vocabulary,
        target_vocabulary,
        source_tokenizer,
        target_tokenizer,
        metadata={
            "epoch": metrics.epoch,
            "validation_loss": metrics.validation_loss,
            "training_config": asdict(training_config),
        },
    )

history = train_model(
    model,
    source_sequences,
    target_sequences,
    source_vocabulary,
    target_vocabulary,
    training_config,
    on_improvement=save_best,
)
history[-1]

## Translate and evaluate a small held-out sample

In [ ]:
translator = Translator.from_checkpoint(checkpoint_path, DEVICE)
sample_sources = test_english[:100]
sample_references = test_french[:100]
sample_hypotheses = [translator.translate(text) for text in sample_sources]
translation_scores(sample_hypotheses, sample_references)